# 04 -- Phase 3: Photometry & Zero Point

Detect sources, measure them with the ERR plane, derive a filter-wise zero
point, flux-calibrate to Janskys, and write an error-carrying catalog.

The zero point is calibrated by cross-matching against an online reference
catalog in a priority cascade: **APASS -> Pan-STARRS -> SDSS** (first hit
wins). This step needs outbound internet on the node.

In [ ]:
# ============================================================
#  WORKSHOP CONFIG  --  EDIT THESE PATHS FOR YOUR ENVIRONMENT
# ============================================================
import os

# 1) Shared, read-only Astrometry.net index directory (used by Phase 2):
os.environ["CASSA_ASTROMETRY_INDEX"] = os.path.abspath("../../astrometry_data")

# 2) The provided workshop dataset + a writable work directory:
RAW_DIR   = "../raw"     # provided raw frames, one level up from notebooks/
WORK_DIR  = "../work"    # writable output dir, one level up from notebooks/

RAW_DIR   = os.path.abspath(os.path.expandvars(RAW_DIR))
WORK_DIR  = os.path.abspath(os.path.expandvars(WORK_DIR))

# These are relative to the working directory, which Jupyter sets to this
# notebook's folder. Fail loudly here rather than confusingly further down.
assert os.path.isdir(RAW_DIR), (
    f"RAW_DIR not found: {RAW_DIR}\nRun this notebook from the notebooks/ "
    f"directory, or set RAW_DIR/WORK_DIR to absolute paths above."
)

# Each phase writes into its own directory under WORK_DIR.
PHASE1_DIR = os.path.join(WORK_DIR, "phase1")   # calibrated frames
PHASE2_DIR = os.path.join(WORK_DIR, "phase2")   # master stacks + WCS
PHASE3_DIR = os.path.join(WORK_DIR, "phase3")   # flux-calibrated + catalogs
PHASE4_DIR = os.path.join(WORK_DIR, "phase4")   # diagnostics report
os.makedirs(WORK_DIR, exist_ok=True)
print("RAW_DIR    =", RAW_DIR)
print("PHASE1_DIR =", PHASE1_DIR)
print("PHASE2_DIR =", PHASE2_DIR)

In [ ]:
import glob, os
assert os.path.isdir(PHASE2_DIR), "No phase2 directory -- run notebook 03 (Phase 2) first."
masters = sorted(glob.glob(os.path.join(PHASE2_DIR, "Master_*.fits")))
assert masters, "No Master_*.fits in PHASE2_DIR -- run notebook 03 (Phase 2) first."
print(f"{len(masters)} master stack(s) in {PHASE2_DIR}")

## Run Phase 3

In [ ]:
from cassa_photometry.config import load_config
from cassa_photometry.phase3_photometry import run as run_p3
cfg = load_config()
run_p3(PHASE2_DIR, default_band='R', outdir=PHASE3_DIR, config=cfg)

## The zero point lives in the flux-calibrated header

In [ ]:
import glob, os
from astropy.io import fits
fc = glob.glob(os.path.join(PHASE3_DIR, '*_fluxcal.fits'))[0]
h = fits.getheader(fc)
for k in ['MAGZERO', 'MAGZERR', 'NZPSTARS', 'FLUXCAL', 'BUNIT']:
    print(f'{k:>9} = {h.get(k)}')

## Explore the source catalog

In [ ]:
import glob, os, pandas as pd
cat = pd.read_csv(glob.glob(os.path.join(PHASE3_DIR, '*_catalog.csv'))[0])
print(len(cat), 'sources')
cat.head()

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].scatter(cat['Absolute_Mag'], cat['Mag_Error'], s=6, alpha=0.4)
ax[0].set_xlabel('magnitude'); ax[0].set_ylabel('Mag_Error'); ax[0].set_title('error vs mag')
ax[1].scatter(cat['Absolute_Mag'], cat['SNR'], s=6, alpha=0.4)
ax[1].set_yscale('log'); ax[1].set_xlabel('magnitude'); ax[1].set_ylabel('SNR')
ax[1].set_title('SNR vs mag')
plt.tight_layout(); plt.show()

## (Optional) Independent verification against reference catalogs

In [ ]:
from cassa_photometry.phase3_photometry.verify import run as verify_run
verify_run(PHASE3_DIR)

### Exercise 1 -- how faint can you trust a magnitude?
Select the round, bright stars (`Ellipticity < 0.15`) and histogram their
`Mag_Error`. Below what magnitude does the error stay under 0.05 mag?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

THRESH = 0.05
stars = cat[cat['Ellipticity'] < 0.15]
print(f"{len(stars)} round stars (Ellipticity < 0.15) out of {len(cat)} sources")

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))

# Left: the histogram the exercise asks for
ax[0].hist(stars['Mag_Error'], bins=30, color='steelblue', edgecolor='white')
ax[0].axvline(THRESH, color='crimson', ls='--', label=f'{THRESH} mag')
ax[0].set_xlabel('Mag_Error'); ax[0].set_ylabel('count')
ax[0].set_title('Mag_Error distribution (round stars)')
ax[0].legend()

# Right: where the error crosses the threshold
bins = np.arange(np.floor(stars['Absolute_Mag'].min()), np.ceil(stars['Absolute_Mag'].max()) + 0.5, 0.5)
binned = stars.groupby(pd.cut(stars['Absolute_Mag'], bins), observed=True)['Mag_Error'].median()
centres = np.array([iv.mid for iv in binned.index])

ax[1].scatter(stars['Absolute_Mag'], stars['Mag_Error'], s=8, alpha=0.4, label='stars')
ax[1].plot(centres, binned.values, 'o-', color='darkorange', label='binned median')
ax[1].axhline(THRESH, color='crimson', ls='--', label=f'{THRESH} mag')
ax[1].set_xlabel('magnitude'); ax[1].set_ylabel('Mag_Error')
ax[1].set_title('error vs magnitude')
ax[1].legend()
plt.tight_layout(); plt.show()

# Answer: the faintest bin whose median error is still under the threshold
under = binned[binned < THRESH]
if len(under) > 0:
    limit = under.index[-1].right
    print(f"\nBinned median Mag_Error stays under {THRESH} mag for stars brighter than ~{limit:.1f} mag")
    print(f"(faintest bin under threshold: {under.index[-1]}, median {under.iloc[-1]:.4f})")
else:
    print(f"\nNo magnitude bin has a median error under {THRESH} mag.")

### Exercise 2 -- check the zero-point arithmetic yourself
The lecture gives two formulas:
$m = -2.5\log_{10}(F) + \mathrm{ZP}$ and
$\delta m = \sqrt{(1.0857\,\delta F/F)^2 + \mathrm{MAGZERR}^2}$.
Recompute `Absolute_Mag` and `Mag_Error` from `Instrumental_Flux`,
`Flux_Error` and the header keywords, and see how well they reproduce the
catalog. Then work out which of the two error terms dominates for the
brightest star, and which for the faintest.

**Careful:** a zero point belongs to one specific image, so pair the catalog
with *its own* `_fluxcal.fits` by base name -- the two `glob` calls earlier in
this notebook can easily land on different filters.

In [ ]:
import glob, os
import numpy as np
import pandas as pd
from astropy.io import fits

# A zero point belongs to ONE image, so pair the catalog with its own fluxcal
# header by base name -- the two globs earlier in this notebook can easily land
# on different filters.
cat_path = sorted(glob.glob(os.path.join(PHASE3_DIR, '*_catalog.csv')))[0]
base = os.path.basename(cat_path).replace('_catalog.csv', '')
fc_path = os.path.join(PHASE3_DIR, base + '_fluxcal.fits')
paired = pd.read_csv(cat_path)
ph = fits.getheader(fc_path)

MAGZERO, MAGZERR = ph['MAGZERO'], ph['MAGZERR']
print(f"Catalog : {os.path.basename(cat_path)}")
print(f"Image   : {os.path.basename(fc_path)}")
print(f"Filter  : {ph.get('FILTER')}   MAGZERO = {MAGZERO:.4f} +/- {MAGZERR:.4f}  "
      f"(from {ph.get('NZPSTARS')} stars)\n")

flux, flux_err = paired['Instrumental_Flux'], paired['Flux_Error']

# 1) m = -2.5 log10(F) + ZP
mag_predicted = -2.5 * np.log10(flux) + MAGZERO
resid_mag = paired['Absolute_Mag'] - mag_predicted

# 2) dm = sqrt( (1.0857 dF/F)^2 + MAGZERR^2 )
err_predicted = np.sqrt((1.0857 * flux_err / flux) ** 2 + MAGZERR ** 2)
resid_err = paired['Mag_Error'] - err_predicted

print(f"{'check':<44} {'max |residual|':>15}")
print(f"{'Absolute_Mag  vs  -2.5log10(F) + MAGZERO':<44} {np.abs(resid_mag).max():15.2e}")
print(f"{'Mag_Error     vs  sqrt((1.0857 dF/F)^2 + ZPerr^2)':<44} {np.abs(resid_err).max():15.2e}")

# Which term dominates the magnitude error, and where?
shot_term = 1.0857 * flux_err / flux
print(f"\n{'':>26} {'brightest star':>16} {'faintest star':>16}")
order = np.argsort(paired['Absolute_Mag'].values)
for label, series in (('measurement 1.0857 dF/F', shot_term),
                      ('zero point MAGZERR', pd.Series(np.full(len(paired), MAGZERR))),
                      ('total Mag_Error', paired['Mag_Error'])):
    b, f_ = series.values[order[0]], series.values[order[-1]]
    print(f"{label:>26} {b:16.4f} {f_:16.4f}")

print(f"\nFor the brightest stars the {'zero point' if shot_term.values[order[0]] < MAGZERR else 'shot noise'} dominates,")
print("so no amount of extra exposure time helps until the ZP itself improves.")


### Exercise 3 -- how deep did this image actually go?
Histogram the source magnitudes (the "number counts") and plot SNR against
magnitude. Real sky holds far more faint sources than bright ones, so the
counts should keep climbing -- where they turn over instead, you are losing
sources to the noise. Find that turnover, and check whether any detection
falls to SNR = 5.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

mags, snr = paired['Absolute_Mag'].values, paired['SNR'].values

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))

# Number counts: real source counts rise with magnitude until detection fails.
bins = np.arange(np.floor(mags.min()), np.ceil(mags.max()) + 0.5, 0.5)
counts, edges, _ = ax[0].hist(mags, bins=bins, color='steelblue', edgecolor='white')
centres = 0.5 * (edges[:-1] + edges[1:])
turnover = centres[np.argmax(counts)]
ax[0].axvline(turnover, color='crimson', ls='--', label=f'turnover ~ {turnover:.2f} mag')
ax[0].set_xlabel('magnitude'); ax[0].set_ylabel('sources per 0.5 mag')
ax[0].set_title('number counts'); ax[0].legend()

# 5-sigma depth: the magnitude at which SNR drops through 5.
ax[1].scatter(mags, snr, s=8, alpha=0.4)
ax[1].axhline(5, color='crimson', ls='--', label='SNR = 5')
ax[1].set_yscale('log'); ax[1].set_xlabel('magnitude'); ax[1].set_ylabel('SNR')
ax[1].set_title('SNR vs magnitude'); ax[1].legend()
plt.tight_layout(); plt.show()

# Interpolate the SNR=5 crossing (SNR falls as magnitude rises, so sort by mag).
order = np.argsort(mags)
m_sorted, snr_sorted = mags[order], snr[order]
below = np.where(snr_sorted < 5)[0]
if len(below) > 0:
    depth = np.interp(-5.0, -snr_sorted[::-1], m_sorted[::-1])
    print(f"5-sigma depth (SNR crosses 5) : {depth:.2f} mag")
else:
    print(f"Every detected source has SNR > 5; faintest is {m_sorted[-1]:.2f} mag "
          f"at SNR {snr_sorted[-1]:.1f}")

print(f"Number-count turnover         : {turnover:.2f} mag")
print(f"Faintest detected source      : {mags.max():.2f} mag")
print(f"Brightest detected source     : {mags.min():.2f} mag")
print(f"Total sources                 : {len(mags)}")

print("\nReal sky has more faint galaxies than bright ones, so the counts should keep")
print("rising. Where they turn over instead, you are losing sources to the noise --")
print("that peak is a completeness limit of THIS image, not a property of the sky.")
